# Epochs Comparison

This notebook compares HAR quantum transformer training behavior across different epoch settings. Datasets are excluded from this repository; configure `DATA_DIR = "../data"` and write outputs to `RESULTS_DIR = "../results"` before running locally.


In [ ]:
#!/usr/bin/env python3
"""
HAR with QTModel under different training-epoch configurations on WISDM / UCI HAR.

QTModel:
 - Strong 1D CNN backbone (64-d features)
 - Stacked quantum blocks:
     * Linear -> angles for Q/K/V
     * TorchQuantum QFM for Q, K, V
     * Self-attention after each QFM block (element-wise Q·K gating of V)
     * Transformer-style FFN + LayerNorm with residuals
 - Final quantum gate (extra QFM) on top of last features
 - Fused [features + gate] -> MLP classifier

Experiment:
 - Fix 4 qubits (wires) and quantum layers = 4
 - Vary number of training epochs:
   E ∈ {50, 100, 150, 200, 250, 500}

Outputs:
 - 6-fold CV metrics for each configuration
 - Training Loss Curves (mean over folds)
 - ROC curves (Validation, AUC)
 - Classification HeatMaps (confusion matrices in %, per configuration)
"""

import os
import urllib.request
from pathlib import Path
import zipfile

import psutil
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader, Subset
from torch.optim.lr_scheduler import CosineAnnealingLR

from ptflops import get_model_complexity_info

from sklearn.model_selection import KFold
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, roc_curve, confusion_matrix
)
from sklearn.preprocessing import label_binarize

import matplotlib.pyplot as plt
from tqdm import trange

import torchquantum as tq

# -----------------------------------------------------------------------------
# GLOBAL CONFIG
# -----------------------------------------------------------------------------

DATASET = "WISDM"   # "WISDM" or "UCIHAR"

QSize = 4                # number of quantum wires (fixed to 4)
Q_LAYERS = 4             # depth of quantum feature map (fixed here)
Q_BLOCKS = 4             # number of quantum blocks
# EPOCHS will be varied per configuration, so we don't use it as a single global.
BATCH_SIZE = 256         # batch size
N_SPLITS = 6             # K-fold CV

WINDOW_SIZE = 200        # default for WISDM, overwritten for UCIHAR if needed
STEP_SIZE = 100          # WISDM step size

DATA_DIR = Path(os.environ.get("DATA_DIR", "../data"))
DATA_DIR.mkdir(exist_ok=True)

# WISDM
WISDM_URL = (
    "https://raw.githubusercontent.com/soham97/WISD_HAR_files/master/"
    "WISDM_ar_v1.1_raw.txt"
)
WISDM_PATH = DATA_DIR / "WISDM_ar_v1.1_raw.txt"

# UCI HAR
UCIHAR_URL = (
    "https://archive.ics.uci.edu/ml/machine-learning-databases/"
    "00240/UCI%20HAR%20Dataset.zip"
)
UCIHAR_ZIP_PATH = DATA_DIR / "UCI_HAR_Dataset.zip"
UCIHAR_EXTRACT_DIR = DATA_DIR / "UCI_HAR_Dataset"

# -----------------------------------------------------------------------------
# 1) DEVICE SELECTION (fixed cuda:0 if available)
# -----------------------------------------------------------------------------

if torch.cuda.is_available():
    DEVICE = torch.device("cuda:0")
    print("Using device: cuda:0")
else:
    DEVICE = torch.device("cpu")
    print("CUDA not available, using CPU")

ram = psutil.virtual_memory()
print(f"RAM Usage: {(ram.total-ram.available)/1e9:.1f}/{ram.total/1e9:.1f} GiB ({ram.percent}%)\n")

# -----------------------------------------------------------------------------
# 2) WISDM & UCI HAR LOADING / PREPROCESSING
# -----------------------------------------------------------------------------

def download_wisdm():
    if WISDM_PATH.exists():
        print(f"[INFO] Found WISDM file at {WISDM_PATH}")
        return
    print(f"[INFO] Downloading WISDM from {WISDM_URL} ...")
    urllib.request.urlretrieve(WISDM_URL, WISDM_PATH)
    print("[INFO] Download complete.")

def load_wisdm_dataframe(path: Path) -> pd.DataFrame:
    """Load and clean WISDM dataset."""
    print("[INFO] Loading WISDM data into DataFrame...")
    columns = ["user", "activity", "timestamp", "x", "y", "z"]
    df = pd.read_csv(
        path,
        header=None,
        names=columns,
        engine="python",
        on_bad_lines="skip",
    )

    df = df.dropna()
    df["z"] = df["z"].astype(str).str.replace(";", "", regex=False)
    df["z"] = df["z"].astype(float)
    df = df[df["timestamp"] != 0]
    df = df.sort_values(by=["user", "timestamp"], ignore_index=True)

    print("[INFO] Cleaned WISDM shape:", df.shape)
    return df

def segment_wisdm(df: pd.DataFrame,
                  window_size: int = WINDOW_SIZE,
                  step_size: int = STEP_SIZE):
    """Segment accelerometer data into windows and compute magnitude."""
    print("[INFO] Segmenting WISDM time series into windows...")
    X_segments = []
    y_labels = []

    users = df["user"].unique()
    for user in users:
        df_user = df[df["user"] == user]
        activities = df_user["activity"].unique()

        for act in activities:
            df_ua = df_user[df_user["activity"] == act]
            signal = df_ua[["x", "y", "z"]].values  # [T, 3]

            for start in range(0, len(signal) - window_size, step_size):
                end = start + window_size
                window = signal[start:end]  # [window_size, 3]
                mag = np.sqrt((window ** 2).sum(axis=1))  # [window_size]
                X_segments.append(mag)
                y_labels.append(act)

    X_segments = np.array(X_segments, dtype=np.float32)
    y_labels = np.array(y_labels)
    print("[INFO] WISDM Segmented X shape:", X_segments.shape)
    print("[INFO] WISDM Label distribution (raw):\n", pd.Series(y_labels).value_counts())
    return X_segments, y_labels

def download_uci_har():
    if UCIHAR_EXTRACT_DIR.exists():
        print(f"[INFO] Found UCI HAR at {UCIHAR_EXTRACT_DIR}")
        return
    if not UCIHAR_ZIP_PATH.exists():
        print(f"[INFO] Downloading UCI HAR from {UCIHAR_URL} ...")
        urllib.request.urlretrieve(UCIHAR_URL, UCIHAR_ZIP_PATH)
        print("[INFO] Download complete.")
    print(f"[INFO] Extracting UCI HAR zip to {UCIHAR_EXTRACT_DIR} ...")
    with zipfile.ZipFile(UCIHAR_ZIP_PATH, "r") as zf:
        zf.extractall(UCIHAR_EXTRACT_DIR)
    print("[INFO] Extraction complete.")

def load_uci_har_magnitude():
    """
    Use body_acc_x/y/z from UCI HAR to build magnitude windows:
      - Each window length = 128
      - Combine train + test
      - Labels: 6 activities (0..5)
    """
    base = UCIHAR_EXTRACT_DIR / "UCI HAR Dataset"

    def load_split(split: str):
        split_dir = base / split
        sig_dir = split_dir / "Inertial Signals"

        # Each: [N_windows, 128]
        x = np.loadtxt(sig_dir / f"body_acc_x_{split}.txt")
        y = np.loadtxt(sig_dir / f"body_acc_y_{split}.txt")
        z = np.loadtxt(sig_dir / f"body_acc_z_{split}.txt")

        mag = np.sqrt(x**2 + y**2 + z**2).astype(np.float32)  # [N, 128]

        # Labels are 1..6 -> 0..5
        y_lab = np.loadtxt(split_dir / f"y_{split}.txt").astype(int) - 1
        return mag, y_lab

    X_train, y_train = load_split("train")
    X_test,  y_test  = load_split("test")

    X_all = np.concatenate([X_train, X_test], axis=0)
    y_all = np.concatenate([y_train, y_test], axis=0)

    print("[INFO] UCI HAR magnitude shape:", X_all.shape)
    print("[INFO] UCI HAR label distribution:",
          dict(zip(*np.unique(y_all, return_counts=True))))
    return X_all, y_all

def balance_by_min_class(X: np.ndarray, y_int: np.ndarray, seed: int = 42):
    """
    Downsample all classes to the smallest class size.
    """
    rng = np.random.default_rng(seed)
    unique_classes = np.unique(y_int)
    indices_per_class = {c: np.where(y_int == c)[0] for c in unique_classes}
    class_sizes = {c: len(idx) for c, idx in indices_per_class.items()}
    min_count = min(class_sizes.values())

    print("[INFO] Class sizes before balancing:", class_sizes)
    print("[INFO] Using min_count =", min_count, "samples per class")

    balanced_indices_list = []
    for c in unique_classes:
        idxs = indices_per_class[c]
        if len(idxs) > min_count:
            chosen = rng.choice(idxs, size=min_count, replace=False)
        else:
            chosen = idxs
        balanced_indices_list.append(chosen)

    balanced_indices = np.concatenate(balanced_indices_list)
    rng.shuffle(balanced_indices)

    balanced_counts = np.bincount(y_int[balanced_indices])
    print("[INFO] Class sizes after balancing:",
          dict(zip(unique_classes, balanced_counts)))
    return X[balanced_indices], y_int[balanced_indices]

# -----------------------------------------------------------------------------
# 3) DATASET CLASS
# -----------------------------------------------------------------------------

class WISDMDataset(Dataset):
    """
    Generic window-level dataset for time-series magnitude signals.
    Each sample: x -> [1, L], y -> integer label.
    """

    def __init__(self, X_windows: np.ndarray, y_int: np.ndarray):
        X_tensor = torch.tensor(X_windows, dtype=torch.float32)
        y_tensor = torch.tensor(y_int, dtype=torch.long)

        # Per-window normalization
        means = X_tensor.mean(dim=1, keepdim=True)
        stds = X_tensor.std(dim=1, keepdim=True) + 1e-8
        X_tensor = (X_tensor - means) / stds

        self.X = X_tensor
        self.y = y_tensor

    def __len__(self):
        return self.y.shape[0]

    def __getitem__(self, idx):
        x = self.X[idx].unsqueeze(0)  # [1, L]
        y = self.y[idx]
        return x, y

# -----------------------------------------------------------------------------
# 4) STRONGER CNN BACKBONE (for QTModel)
# -----------------------------------------------------------------------------

class EnhancedCNN1D(nn.Module):
    """
    Strong 1D CNN backbone for QTModel.
    Input:  [B, 1, L]
    Output: [B, 64]
    """
    def __init__(self, in_ch: int = 1):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv1d(in_ch, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2)   # L -> L/2
        )
        self.block2 = nn.Sequential(
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2)   # L/2 -> L/4
        )
        self.block3 = nn.Sequential(
            nn.Conv1d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)  # [B,64,1]
        )
        self.dropout = nn.Dropout(p=0.2)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.dropout(x)
        return x.flatten(1)  # [B, 64]

# -----------------------------------------------------------------------------
# 5) TORCHQUANTUM FEATURE MAP
# -----------------------------------------------------------------------------

class QuantumFeatureMapTQ(tq.QuantumModule):
    """
    TorchQuantum-based feature map:
     - Angle embedding via RY
     - Ring entangling CNOT chain
     - Measure all Z
    """
    def __init__(self, n_wires=4, n_layers=4):
        super().__init__()
        self.n_wires  = n_wires
        self.n_layers = n_layers
        self.measure  = tq.MeasureAll(tq.PauliZ)

    def forward(self, qdev, angles):
        # angles: (batch, n_wires)
        for w in range(self.n_wires):
            qdev.ry(wires=w, params=angles[:, w], static=self.static_mode)
        for _ in range(self.n_layers):
            for i in range(self.n_wires - 1):
                qdev.cnot(wires=[i, i+1])
            qdev.cnot(wires=[self.n_wires-1, 0])
        return self.measure(qdev)  # (batch, n_wires)

# -----------------------------------------------------------------------------
# 6) QUANTUM BLOCK WITH SELF-ATTENTION + FFN
# -----------------------------------------------------------------------------

class QBlockQT(tq.QuantumModule):
    """
    Single quantum block for QTModel:

      Input:  x ∈ R^{B×D}
      Steps:
        - Linear -> angles for Q, K, V (each in R^{B×n_wires})
        - QFM for Q, K, V (TorchQuantum)
        - Self-attention:
            q = Wq(q_out), k = Wk(k_out), v = Wv(v_out) ∈ R^{B×D}
            scores = softmax( (q * k) / sqrt(D) )
            attn_out = scores * v
        - Residual: z = x + attn_out
        - LayerNorm + Transformer-style FFN + residual + LayerNorm
    """
    def __init__(self, dim=64, n_wires=4, n_layers=4, ff_mult=2, dropout=0.1):
        super().__init__()
        self.dim     = dim
        self.n_wires = n_wires

        # QFM for Q, K, V
        self.q_fm = QuantumFeatureMapTQ(n_wires, n_layers)
        self.k_fm = QuantumFeatureMapTQ(n_wires, n_layers)
        self.v_fm = QuantumFeatureMapTQ(n_wires, n_layers)

        # Map input features -> Q/K/V angles
        self.reduce = nn.Linear(dim, 3 * n_wires)

        # Project Q/K/V outputs up to feature dim
        self.q_proj = nn.Linear(n_wires, dim)
        self.k_proj = nn.Linear(n_wires, dim)
        self.v_proj = nn.Linear(n_wires, dim)

        # Transformer-style FFN
        self.ff = nn.Sequential(
            nn.Linear(dim, ff_mult * dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_mult * dim, dim),
        )

        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)

    def forward(self, x):
        # x: [B, D]
        B, D = x.shape

        # Classical to quantum angles
        r = self.reduce(x)                      # [B, 3 * n_wires]
        q_ang, k_ang, v_ang = r.chunk(3, dim=1) # each [B, n_wires]

        # Quantum device
        qdev = tq.QuantumDevice(
            n_wires=self.n_wires,
            bsz=B,
            device=x.device
        )

        # QFM for Q, K, V
        q_out = self.q_fm(qdev, q_ang)  # [B, n_wires]
        k_out = self.k_fm(qdev, k_ang)
        v_out = self.v_fm(qdev, v_ang)

        # Project to feature dimension
        q = self.q_proj(q_out)  # [B, D]
        k = self.k_proj(k_out)  # [B, D]
        v = self.v_proj(v_out)  # [B, D]

        # Self-attention (element-wise gating)
        scale = float(D) ** 0.5
        scores = torch.softmax((q * k) / scale, dim=-1)  # [B, D]
        attn_out = scores * v                            # [B, D]

        # Residual + FFN + norms
        z = x + attn_out          # first residual
        z_norm = self.norm1(z)
        ff_out = self.ff(z_norm)
        y = self.norm2(z_norm + ff_out)  # second residual
        return y

# -----------------------------------------------------------------------------
# 7) QTModel: CNN -> stacked QBlocks -> quantum gate -> classifier
# -----------------------------------------------------------------------------

class QTModel(nn.Module):
    """
    QTModel:
      - EnhancedCNN1D -> 64-d feature
      - Multiple QBlockQT (each with QFM+SelfAttention+FFN)
      - Final quantum gate:
          angles = W_gate(e)
          q_gate = QFM(angles)
          fused = [e, q_gate]
          MLP classifier
    """
    def __init__(self, num_cls, in_ch=1,
                 n_blocks=Q_BLOCKS, n_wires=QSize, n_layers=Q_LAYERS,
                 dim=64, ff_mult=2, dropout=0.1):
        super().__init__()

        self.dim      = dim
        self.n_wires  = n_wires
        self.cnn      = EnhancedCNN1D(in_ch)

        # Stacked quantum blocks
        self.blocks = nn.ModuleList([
            QBlockQT(dim=dim, n_wires=n_wires, n_layers=n_layers,
                     ff_mult=ff_mult, dropout=dropout)
            for _ in range(n_blocks)
        ])

        # Final quantum gate
        self.gate_linear = nn.Linear(dim, n_wires)
        self.gate_qfm    = QuantumFeatureMapTQ(n_wires, n_layers)

        # Classifier on fused [features + quantum gate]
        self.cls_head = nn.Sequential(
            nn.LayerNorm(dim + n_wires),
            nn.Linear(dim + n_wires, dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, num_cls),
        )

    def forward(self, x):
        # CNN backbone
        e = self.cnn(x)  # [B, 64]

        # Quantum blocks
        for block in self.blocks:
            e = block(e)  # [B, 64]

        # Final quantum gate
        angles = self.gate_linear(e)  # [B, n_wires]
        qdev = tq.QuantumDevice(
            n_wires=self.n_wires,
            bsz=angles.size(0),
            device=angles.device
        )
        q_gate = self.gate_qfm(qdev, angles)  # [B, n_wires]

        fused = torch.cat([e, q_gate], dim=1)  # [B, 64 + n_wires]
        logits = self.cls_head(fused)          # raw logits
        return logits

# -----------------------------------------------------------------------------
# 8) BASELINE MODELS (defined but not used in this experiment)
# -----------------------------------------------------------------------------

def conv_dw(in_planes, out_planes, kernel_size, stride=1, dilation=1):
    return nn.Sequential(
        nn.Conv1d(in_planes, in_planes, kernel_size=kernel_size,
                  stride=stride, padding=(kernel_size//2)*dilation, dilation=dilation,
                  groups=in_planes, bias=False),
        nn.Conv1d(in_planes, out_planes, kernel_size=1, bias=False)
    )

class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super().__init__()
        self.avg = nn.AdaptiveAvgPool1d(1)
        self.max = nn.AdaptiveMaxPool1d(1)
        red = max(1, in_planes // ratio)
        self.fc = nn.Sequential(
            nn.Conv1d(in_planes, red, 1, bias=False), nn.ReLU(),
            nn.Conv1d(red, in_planes, 1, bias=False)
        )
        self.sig = nn.Sigmoid()
    def forward(self, x):
        return self.sig(self.fc(self.avg(x)) + self.fc(self.max(x)))

class SpatialAttention(nn.Module):
    def __init__(self, k=7):
        super().__init__()
        self.conv1 = nn.Conv1d(2, 1, k, padding=k//2, bias=False)
        self.sig = nn.Sigmoid()
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        return self.sig(self.conv1(torch.cat([avg_out, max_out], dim=1)))

class BasicBlock1D(nn.Module):
    expansion = 1
    def __init__(self, in_planes, out_planes, k, stride=1, dilation=1, downsample=None):
        super().__init__()
        self.conv1 = conv_dw(in_planes, out_planes, k, stride, dilation)
        self.bn1 = nn.BatchNorm1d(out_planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv_dw(out_planes, out_planes, k)
        self.bn2 = nn.BatchNorm1d(out_planes)
        self.ca = ChannelAttention(out_planes)
        self.sa = SpatialAttention()
        self.downsample = downsample
    def forward(self, x):
        res = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.ca(out)*out
        out = self.sa(out)*out
        if self.downsample is not None:
            res = self.downsample(x)
        return self.relu(out + res)

class DeepILS(nn.Module):
    def __init__(self, num_inputs=1, num_classes=6, block=BasicBlock1D, group_sizes=(2,2,2,2), base=64, k=3):
        super().__init__()
        self.inp = nn.Sequential(
            nn.Conv1d(num_inputs, base, 5, 2, 3, bias=False),
            nn.BatchNorm1d(base),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(3,2,1)
        )
        self.inplanes = base
        self.groups = nn.ModuleList()
        planes = [base*(2**i) for i in range(len(group_sizes))]
        strides = [1] + [2]*(len(group_sizes)-1)
        for i, blocks in enumerate(group_sizes):
            pl = planes[i]
            st = strides[i]
            down = None
            if st != 1 or self.inplanes != pl*block.expansion:
                down = nn.Sequential(
                    nn.Conv1d(self.inplanes, pl*block.expansion, 1, st, bias=False),
                    nn.BatchNorm1d(pl*block.expansion)
                )
            layers = [block(self.inplanes, pl, k, stride=st, downsample=down)]
            self.inplanes = pl*block.expansion
            for _ in range(1, blocks):
                layers.append(block(self.inplanes, pl, k))
            self.groups.append(nn.Sequential(*layers))
        self.groups = nn.Sequential(*self.groups)
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(planes[-1]*block.expansion, num_classes)
        )
        self._init()
    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)
    def forward(self, x):
        x = self.inp(x)
        x = self.groups(x)
        return self.head(x)

class ResNet1D(DeepILS):
    pass

class Swish(nn.Module):
    def forward(self, x): return x*torch.sigmoid(x)

def Conv_3x3(inp, oup, s):
    return nn.Sequential(
        nn.Conv1d(inp,oup,3,s,1,bias=False),
        nn.BatchNorm1d(oup),
        Swish()
    )

def Conv_1x1(inp, oup):
    return nn.Sequential(
        nn.Conv1d(inp,oup,1,1,0,bias=False),
        nn.BatchNorm1d(oup),
        Swish()
    )

class InvertedResidual_Eff(nn.Module):
    def __init__(self, inp, oup, s, t, k):
        super().__init__()
        self.use=(s==1 and inp==oup)
        hid=inp*t
        self.conv=nn.Sequential(
            nn.Conv1d(inp,hid,1,1,0,bias=False), nn.BatchNorm1d(hid), Swish(),
            nn.Conv1d(hid,hid,k,s,k//2,groups=hid,bias=False), nn.BatchNorm1d(hid), Swish(),
            nn.Conv1d(hid,oup,1,1,0,bias=False), nn.BatchNorm1d(oup)
        )
    def forward(self,x):
        y=self.conv(x)
        return x+y if self.use else y

class EfficientNetB0_1D(nn.Module):
    def __init__(self, n_classes=6, wm=1.0):
        super().__init__()
        setting=[[1,16,1,1,3],[6,24,2,2,3],[6,40,2,2,5],
                 [6,80,3,2,3],[6,112,3,1,5],[6,192,4,2,5],[6,320,1,1,3]]
        in_ch=int(32*wm)
        last=1280 if wm<=1 else int(1280*wm)
        feats=[Conv_3x3(1,in_ch,2)]
        ch=in_ch
        for t,c,n,s,k in setting:
            oc=int(c*wm)
            for i in range(n):
                feats.append(InvertedResidual_Eff(ch, oc, s if i==0 else 1, t, k))
                ch=oc
        feats += [Conv_1x1(ch,last), nn.AdaptiveAvgPool1d(1)]
        self.features=nn.Sequential(*feats)
        self.fc=nn.Linear(last,n_classes)
        self._init()
    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight,1); nn.init.constant_(m.bias,0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight,0,0.01); nn.init.constant_(m.bias,0)
    def forward(self,x):
        return self.fc(self.features(x).squeeze(-1))

class InvertedResidual_Mnas(nn.Module):
    def __init__(self, inp, oup, s, t, k):
        super().__init__()
        self.use=(s==1 and inp==oup)
        hid=inp*t
        self.conv=nn.Sequential(
            nn.Conv1d(inp,hid,1,1,0,bias=False), nn.BatchNorm1d(hid), nn.ReLU6(inplace=True),
            nn.Conv1d(hid,hid,k,s,k//2,groups=hid,bias=False), nn.BatchNorm1d(hid), nn.ReLU6(inplace=True),
            nn.Conv1d(hid,oup,1,1,0,bias=False), nn.BatchNorm1d(oup)
        )
    def forward(self,x):
        y=self.conv(x)
        return x+y if self.use else y

def SepConv_3x3(inp,oup):
    return nn.Sequential(
        nn.Conv1d(inp,inp,3,1,1,groups=inp,bias=False), nn.BatchNorm1d(inp), nn.ReLU6(inplace=True),
        nn.Conv1d(inp,oup,1,1,0,bias=False), nn.BatchNorm1d(oup)
    )

def Conv3_3(inp,oup,s):
    return nn.Sequential(
        nn.Conv1d(inp,oup,3,s,1,bias=False), nn.BatchNorm1d(oup), nn.ReLU6(inplace=True)
    )

def Conv1_1(inp,oup):
    return nn.Sequential(
        nn.Conv1d(inp,oup,1,1,0,bias=False), nn.BatchNorm1d(oup), nn.ReLU6(inplace=True)
    )

class MnasNet_1D(nn.Module):
    def __init__(self, n_classes=6, wm=1.0):
        super().__init__()
        setting=[[3,24,3,2,3],[3,40,3,2,5],[6,80,3,2,5],
                 [6,96,2,1,3],[6,192,4,2,5],[6,320,1,1,3]]
        in_ch=int(32*wm)
        last=1280 if wm<=1 else int(1280*wm)
        feats=[Conv3_3(1,in_ch,2), SepConv_3x3(in_ch,16)]
        ch=16
        for t,c,n,s,k in setting:
            oc=int(c*wm)
            for i in range(n):
                feats.append(InvertedResidual_Mnas(ch, oc, s if i==0 else 1, t, k))
                ch=oc
        feats += [Conv1_1(ch,last), nn.AdaptiveAvgPool1d(1)]
        self.features=nn.Sequential(*feats)
        self.fc=nn.Linear(last,n_classes)
        self._init()
    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight,mode='fan_out',nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight,1); nn.init.constant_(m.bias,0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight,0,0.01); nn.init.constant_(m.bias,0)
    def forward(self,x):
        return self.fc(self.features(x).squeeze(-1))

class DSConv_Mobile(nn.Module):
    def __init__(self, f3, f1, s=1, p=1):
        super().__init__()
        self.seq=nn.Sequential(
            nn.Conv1d(f3,f3,3,s,p,groups=f3,bias=False), nn.BatchNorm1d(f3), nn.ReLU(inplace=True),
            nn.Conv1d(f3,f1,1,1,0,bias=False), nn.BatchNorm1d(f1), nn.ReLU(inplace=True)
        )
    def forward(self,x): return self.seq(x)

class MobileNet_1D(nn.Module):
    def __init__(self, channels=[32,64,128,256,512,1024], wm=1.0, n_classes=6):
        super().__init__()
        ch=[int(c*wm) for c in channels]
        self.conv=nn.Sequential(
            nn.Conv1d(1,ch[0],3,2,1,bias=False), nn.BatchNorm1d(ch[0]), nn.ReLU(inplace=True)
        )
        self.features=nn.Sequential(
            DSConv_Mobile(ch[0],ch[1],1,1),
            DSConv_Mobile(ch[1],ch[2],2,1),
            DSConv_Mobile(ch[2],ch[2],1,1),
            DSConv_Mobile(ch[2],ch[3],2,1),
            DSConv_Mobile(ch[3],ch[3],1,1),
            DSConv_Mobile(ch[3],ch[4],2,1),
            DSConv_Mobile(ch[4],ch[4],1,1),
            DSConv_Mobile(ch[4],ch[4],1,1),
            DSConv_Mobile(ch[4],ch[4],1,1),
            DSConv_Mobile(ch[4],ch[4],1,1),
            DSConv_Mobile(ch[4],ch[4],1,1),
            DSConv_Mobile(ch[4],ch[5],2,1),
            DSConv_Mobile(ch[5],ch[5],1,1),
        )
        self.avg=nn.AdaptiveAvgPool1d(1)
        self.fc=nn.Linear(ch[5],n_classes)
    def forward(self,x):
        x=self.conv(x)
        x=self.features(x)
        x=self.avg(x).squeeze(-1)
        return self.fc(x)

class InvertedResidual_V2(nn.Module):
    def __init__(self, inp, oup, s, t):
        super().__init__()
        hid=int(inp*t)
        self.use=(s==1 and inp==oup)
        if t==1:
            self.conv=nn.Sequential(
                nn.Conv1d(hid,hid,3,s,1,groups=hid,bias=False), nn.BatchNorm1d(hid), nn.ReLU6(inplace=True),
                nn.Conv1d(hid,oup,1,1,0,bias=False), nn.BatchNorm1d(oup)
            )
        else:
            self.conv=nn.Sequential(
                nn.Conv1d(inp,hid,1,1,0,bias=False), nn.BatchNorm1d(hid), nn.ReLU6(inplace=True),
                nn.Conv1d(hid,hid,3,s,1,groups=hid,bias=False), nn.BatchNorm1d(hid), nn.ReLU6(inplace=True),
                nn.Conv1d(hid,oup,1,1,0,bias=False), nn.BatchNorm1d(oup)
            )
    def forward(self,x):
        y=self.conv(x)
        return x+y if self.use else y

def make_divisible(x, d=8):
    return int((x + d - 1) // d * d)

class MobileNetV2_1D(nn.Module):
    def __init__(self, n_classes=6, wm=1.0):
        super().__init__()
        input_channel=32
        last=1280
        setting=[[1,16,1,1],[6,24,2,2],[6,32,3,2],[6,64,4,2],
                 [6,96,3,1],[6,160,3,2],[6,320,1,1]]
        self.last_ch=make_divisible(last*wm) if wm>1 else last
        feats=[nn.Conv1d(1,input_channel,3,2,1,bias=False), nn.BatchNorm1d(input_channel), nn.ReLU6(inplace=True)]
        ch=input_channel
        for t,c,n,s in setting:
            oc=make_divisible(c*wm) if t>1 else c
            for i in range(n):
                feats.append(InvertedResidual_V2(ch, oc, s if i==0 else 1, t))
                ch=oc
        feats += [nn.Conv1d(ch,self.last_ch,1,1,0,bias=False), nn.BatchNorm1d(self.last_ch), nn.ReLU6(inplace=True)]
        self.features=nn.Sequential(*feats)
        self.pool=nn.AdaptiveAvgPool1d(1)
        self.fc=nn.Linear(self.last_ch,n_classes)
        self._init()
    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight,mode='fan_out',nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight,1); nn.init.constant_(m.bias,0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight,0,0.01); nn.init.constant_(m.bias,0)
    def forward(self,x):
        x=self.features(x)
        x=self.pool(x).squeeze(-1)
        return self.fc(x)

# --- TSLANet (compact) ---
class TSLANet(nn.Module):
    def __init__(self, C_in, T_in, n_classes=6, emb=128, depth=3, patch=8, drop=0.10):
        super().__init__()
        stride=max(1, patch//2)
        self.proj=nn.Conv1d(C_in,emb,patch,stride)
        N=int((T_in-patch)/stride+1)
        self.pos=nn.Parameter(torch.zeros(1,N,emb))
        self.blocks=nn.ModuleList([
            nn.Sequential(
                nn.LayerNorm(emb),
                nn.Identity(),  # spectral placeholder
                nn.LayerNorm(emb),
                nn.Sequential(
                    nn.Linear(emb,int(emb*3)), nn.GELU(),
                    nn.Linear(int(emb*3),emb)
                )
            ) for _ in range(depth)
        ])
        self.drop=drop
        self.head=nn.Sequential(
            nn.Dropout(drop),
            nn.Linear(emb,emb//2),
            nn.GELU(),
            nn.Linear(emb//2,n_classes)
        )
    def forward(self,x):
        x=self.proj(x).flatten(2).transpose(1,2)
        x=x+self.pos
        x=F.dropout(x,p=self.drop,training=self.training)
        for b in self.blocks:
            x=x + b[3](b[1](b[0](x)))
        return self.head(x.mean(1))

# -----------------------------------------------------------------------------
# 9) TRAIN / EVAL HELPERS
# -----------------------------------------------------------------------------

def process_target(y: torch.Tensor) -> torch.Tensor:
    if y.dim() > 1:
        if y.size(1) > 1:
            y = y.argmax(dim=1)
        else:
            y = y.squeeze(1)
    return y.long()

def train_one(model, loader, opt):
    model.train()
    losses = []
    for x, y in loader:
        x = x.to(DEVICE)
        y = process_target(y.to(DEVICE))

        out = model(x)               # logits
        loss = F.cross_entropy(out, y)

        opt.zero_grad()
        loss.backward()
        opt.step()

        losses.append(loss.item())
    return float(np.mean(losses)) if losses else 0.0

def eval_fold(model, loader, num_cls):
    """Evaluate on one fold: return y_true and probabilities."""
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            y = process_target(y.to(DEVICE))
            out  = model(x)  # logits
            prob = F.softmax(out, dim=1).cpu().numpy()
            ys.append(y.cpu().numpy())
            ps.append(prob)
    ys = np.concatenate(ys)
    ps = np.concatenate(ps)
    return ys, ps

def run_cv_for_model(model_name, model_ctor, num_cls, dataset, num_epochs):
    """
    Run K-fold CV for a given model configuration and number of epochs.
    """
    N = len(dataset)
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=0)

    metrics = {"acc": [], "f1": [], "prec": [], "rec": [], "auc": []}
    loss_hist = []
    all_y = []
    all_probs = []

    print(f"\n====== Running 6-fold CV for {model_name} (epochs={num_epochs}) ======")

    for fold, (train_idx, val_idx) in enumerate(kf.split(np.arange(N)), start=1):
        print(f"\n--- {model_name} Fold {fold}/{N_SPLITS} ---")

        train_loader = DataLoader(
            Subset(dataset, train_idx),
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=4,
            pin_memory=True
        )
        val_loader = DataLoader(
            Subset(dataset, val_idx),
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=4,
            pin_memory=True
        )

        model = model_ctor().to(DEVICE)
        opt   = optim.Adam(model.parameters(), lr=3e-3, weight_decay=1e-4)
        sch   = CosineAnnealingLR(opt, T_max=num_epochs)

        fold_losses = []
        for _ in trange(num_epochs, desc=f"{model_name}-F{fold}", leave=False):
            loss = train_one(model, train_loader, opt)
            sch.step()
            fold_losses.append(loss)
        loss_hist.append(fold_losses)

        y_fold, p_fold = eval_fold(model, val_loader, num_cls)
        preds = p_fold.argmax(axis=1)

        acc  = accuracy_score(y_fold, preds)
        f1   = f1_score(y_fold, preds, average="weighted")
        prec = precision_score(y_fold, preds, average="weighted")
        rec  = recall_score(y_fold, preds, average="weighted")

        if num_cls == 2:
            auc_val = roc_auc_score(y_fold, p_fold[:,1])
        else:
            y_bin = label_binarize(y_fold, classes=list(range(num_cls)))
            auc_val = roc_auc_score(y_bin, p_fold, average="micro", multi_class="ovo")

        print(f" Fold {fold}: Acc={acc:.4f} F1={f1:.4f} Prec={prec:.4f} Rec={rec:.4f} AUC={auc_val:.4f}")
        for k,v in zip(("acc","f1","prec","rec","auc"), (acc,f1,prec,rec,auc_val)):
            metrics[k].append(v)

        all_y.append(y_fold)
        all_probs.append(p_fold)

        # free model this fold
        del model
        torch.cuda.empty_cache()

    all_y = np.concatenate(all_y)
    all_probs = np.concatenate(all_probs, axis=0)

    # Global ROC for the model
    if num_cls == 2:
        fpr, tpr, _ = roc_curve(all_y, all_probs[:,1], pos_label=1)
        auc_global = roc_auc_score(all_y, all_probs[:,1])
    else:
        y_bin = label_binarize(all_y, classes=list(range(num_cls)))
        fpr, tpr, _ = roc_curve(y_bin.ravel(), all_probs.ravel())
        auc_global = roc_auc_score(y_bin, all_probs, average="micro", multi_class="ovo")

    return metrics, loss_hist, all_y, all_probs, (fpr, tpr, auc_global)

# -----------------------------------------------------------------------------
# 10) MAIN: DATA CHOICE + MODELS + PLOTS
# -----------------------------------------------------------------------------

def main():
    global WINDOW_SIZE

    # -----------------------
    # Dataset choice
    # -----------------------
    if DATASET == "WISDM":
        download_wisdm()
        df = load_wisdm_dataframe(WISDM_PATH)
        X_segments, y_labels = segment_wisdm(df)

        labels_cat = pd.Series(y_labels).astype("category")
        class_names = list(labels_cat.cat.categories)
        y_int = labels_cat.cat.codes.to_numpy().astype(np.int64)

        WINDOW_SIZE = X_segments.shape[1]

    elif DATASET == "UCIHAR":
        download_uci_har()
        X_segments, y_int = load_uci_har_magnitude()

        class_names = [
            "WALKING",
            "WALK_UP",
            "WALK_DOWN",
            "SITTING",
            "STANDING",
            "LAYING",
        ]
        WINDOW_SIZE = X_segments.shape[1]

    else:
        raise ValueError(f"Unknown DATASET: {DATASET}")

    print("[INFO] Using dataset:", DATASET)
    print("[INFO] Classes:", class_names)

    # Balance dataset
    X_bal, y_bal = balance_by_min_class(X_segments, y_int, seed=42)
    dataset = WISDMDataset(X_bal, y_bal)
    N = len(dataset)
    print(f"[INFO] Balanced dataset size: {N}")

    num_cls = len(class_names)
    seq_len = X_bal.shape[1]
    dims = (1, seq_len)

    # ------------------------
    # QTModel with different epoch configurations (4 qubits fixed)
    # ------------------------
    epoch_configs = [50, 100, 150, 200, 250, 500]

    # All use same architecture; only training length differs
    model_factories = {
        f"QTModel_E{E}": (lambda E=E: QTModel(num_cls=num_cls, in_ch=1,
                                             n_wires=QSize, n_layers=Q_LAYERS))
        for E in epoch_configs
    }

    # ------------------------
    # Complexity (params/FLOPs) for each configuration (same for all, but computed per name)
    # ------------------------
    complexity = {}
    for name, ctor in model_factories.items():
        model = ctor().to(DEVICE)
        try:
            macs, params = get_model_complexity_info(
                model, dims,
                as_strings=True,
                print_per_layer_stat=False
            )
        except Exception as e:
            print(f"[WARN] ptflops failed on {name}: {e}")
            macs, params = "N/A", "N/A"
        complexity[name] = (params, macs)
        del model
        torch.cuda.empty_cache()
        print(f"[INFO] {name}: params={params}, FLOPs={macs}")

    # ------------------------
    # Run CV for each epoch configuration
    # ------------------------
    all_metrics  = {}
    all_losses   = {}
    all_yprobs   = {}
    all_rocs     = {}

    for E in epoch_configs:
        name = f"QTModel_E{E}"
        ctor = model_factories[name]
        metrics, loss_hist, y_all, p_all, roc_info = run_cv_for_model(
            name, ctor, num_cls, dataset, num_epochs=E
        )
        all_metrics[name] = metrics
        all_losses[name]  = loss_hist
        all_yprobs[name]  = (y_all, p_all)
        all_rocs[name]    = roc_info

    # Summary table
    gpu_mem = None
    if DEVICE.type == "cuda":
        gpu_mem = (torch.cuda.memory_allocated()/1e6,
                   torch.cuda.memory_reserved()/1e6)

    summary_rows = []
    for E in epoch_configs:
        name = f"QTModel_E{E}"
        m = all_metrics[name]
        params, macs = complexity.get(name, ("N/A", "N/A"))

        row = {
            "model":        name,
            "epochs":       E,
            "dataset":      DATASET,
            "acc_mean±sd":  f"{np.mean(m['acc']):.4f}±{np.std(m['acc']):.4f}",
            "f1_mean±sd":   f"{np.mean(m['f1']):.4f}±{np.std(m['f1']):.4f}",
            "prec_mean±sd": f"{np.mean(m['prec']):.4f}±{np.std(m['prec']):.4f}",
            "rec_mean±sd":  f"{np.mean(m['rec']):.4f}±{np.std(m['rec']):.4f}",
            "auc_mean±sd":  f"{np.mean(m['auc']):.4f}±{np.std(m['auc']):.4f}",
            "GPU_mem_MB":   gpu_mem,
            "params":       params,
            "FLOPs":        macs,
        }
        summary_rows.append(row)

    df_sum = pd.DataFrame(summary_rows)
    print("\n[SUMMARY COMPARISON (QTModel vs Epochs)]")
    print(df_sum.to_markdown(index=False))

    # -------------------------------------------------
    # PLOTS
    # -------------------------------------------------

    # 1) Training Loss Curves (mean over folds) per epoch configuration
    plt.figure()
    for name, loss_hist in all_losses.items():
        loss_arr = np.array(loss_hist)  # [n_folds, epochs_for_that_config]
        mean_loss = loss_arr.mean(axis=0)
        epochs_len = loss_arr.shape[1]
        plt.plot(range(1, epochs_len+1), mean_loss, label=name)
    plt.xlabel("Epoch")
    plt.ylabel("Training Loss")
    plt.title(f"QTModel Training Loss vs Epoch Configs - {DATASET}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    # 2) ROC curves per configuration (Validation)
    plt.figure()
    for name, (fpr, tpr, auc_val) in all_rocs.items():
        plt.plot(fpr, tpr, label=f"{name} (AUC={auc_val:.4f})")
    plt.plot([0,1],[0,1],'k--',alpha=0.5)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"QTModel ROC Curves (Validation, all folds aggregated) - {DATASET}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    # 3) Classification HeatMaps (Confusion Matrices in %)
    num_cls = len(class_names)
    for name, (y_all, p_all) in all_yprobs.items():
        preds = p_all.argmax(axis=1)
        cm = confusion_matrix(y_all, preds, labels=list(range(num_cls)))
        cm_sum = cm.sum(axis=1, keepdims=True) + 1e-8
        cm_percent = cm.astype(float) / cm_sum * 100.0

        plt.figure(figsize=(6,5))
        im = plt.imshow(cm_percent, interpolation='nearest', cmap="Blues", vmin=0, vmax=100)
        plt.title(f"Confusion Matrix (%) - {name} ({DATASET})")
        plt.colorbar(im, fraction=0.046, pad=0.04)
        tick_marks = np.arange(num_cls)
        plt.xticks(tick_marks, class_names, rotation=45, ha="right")
        plt.yticks(tick_marks, class_names)

        thresh = cm_percent.max() / 2.
        for i in range(num_cls):
            for j in range(num_cls):
                plt.text(j, i, f"{cm_percent[i, j]:.1f}",
                         ha="center", va="center",
                         color="white" if cm_percent[i, j] > thresh else "black",
                         fontsize=8)
        plt.ylabel("True label")
        plt.xlabel("Predicted label")
        plt.tight_layout()
        plt.show()

if __name__ == "__main__":
    main()
